In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
admin_rdm_url = 'http://localhost:8001/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
institution_name = 'Massachusetts Institute of Technology [Test]'
project_name_template = None  # テンプレート登録用プロジェクト
project_name_execution = None  # ワークフロー実行用プロジェクト（アクティベーションあり）
workflow_name = None
engine_name = None
workflow_zip_path = 'resources/simple-approval-workflow.zip'
start_form_content = 'Test request content for lifecycle test'
default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False

In [ ]:
import tempfile

if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')

if project_name_template is None:
    project_name_template = input(prompt='Project name for template registration')
if project_name_execution is None:
    project_name_execution = input(prompt='Project name for workflow execution')
if workflow_name is None:
    workflow_name = input(prompt='Workflow template name')
if engine_name is None:
    engine_name = input(prompt='Workflow engine name')

cascade_test_template_name = datetime.now().strftime('CASCADE-TEST-%Y%m%d%H%M')

# ワークフローライフサイクル管理

- サブシステム名: アドオン
- ページ/アドオン: Workflow
- 機能分類: ライフサイクル管理
- シナリオ名: エンジン・テンプレート・アクティベーションの無効化・有効化・削除
- 用意するテストデータ: アカウント、登録済みエンジン・テンプレート・アクティベーション
- 事前条件: テンプレート登録が完了し、アクティベーションが有効化されていること（実行中ワークフローなし）

In [ ]:
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import asyncio
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## 新規タブを開き、「同意する」をクリックする

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await page.locator('//button[text() = "同意する"]').click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_page=True)

## 「ユーザー名」「パスワード」を入力し「ログイン」をクリックする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧からワークフロー実行用プロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
project_url_execution = None
project_url_template = None

async def _step(page):
    global project_url_execution
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name_execution}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    project_url_execution = page.url

await run_pw(_step)

## 「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('#activationsPanel').locator('//button[*[text() = "無効にする"]]').scroll_into_view_if_needed()
    await expect(page.locator('#activationsPanel').locator('//button[*[text() = "無効にする"]]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## アクティベーション一覧で対象行の無効化ボタンをクリックする

確認ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#activationsPanel').locator('//button[*[text() = "無効にする"]]').click()
    await expect(page.locator('//h4[text() = "ワークフローの無効化"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 確認ダイアログで確認ボタンをクリックする

アクティベーションのステータスが「無効」に変わること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(@data-bind, "confirmDisableActivation")]').click()
    await expect(page.locator('#activationsPanel').locator('//*[contains(@class, "label") and text() = "無効"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトトップに戻る

ワークフロー開始ボタンが非表示であること

In [ ]:
async def _step(page):
    await page.goto(project_url_execution)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('//*[contains(@class, "fa-refresh")]').first).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'#workflow-dashboard').locator(f'//a[contains(@class, "btn-primary") and contains(text(), "{workflow_name}")]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## 「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('#activationsPanel').locator('//button[contains(text(), "有効にする")]').scroll_into_view_if_needed()
    await expect(page.locator('#activationsPanel').locator('//button[contains(text(), "有効にする")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## アクティベーション一覧で対象行の有効化ボタンをクリックする

アクティベーションのステータスが「有効」に変わること

In [ ]:
async def _step(page):
    await page.locator('#activationsPanel').locator('//button[contains(text(), "有効にする")]').click()
    await expect(page.locator('#activationsPanel').locator('//*[contains(@class, "label") and text() = "有効"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトトップに戻る

ワークフロー開始ボタンが表示されること

In [ ]:
async def _step(page):
    await page.goto(project_url_execution)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'#workflow-dashboard').locator(f'//a[contains(@class, "btn-primary") and contains(text(), "{workflow_name}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ワークフローパネルのテンプレートリンクをクリックする

ワークフロー開始フォームが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'#workflow-dashboard').locator(f'//a[contains(@class, "btn-primary") and contains(text(), "{workflow_name}")]').click()
    await expect(page.locator('#workflow-field-confirm_checkbox')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## 「内容を確認しました」をチェックし、「申請内容」を入力する

入力内容が反映されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-field-confirm_checkbox').check()
    await page.locator('#workflow-field-request_content').fill(start_form_content)

await run_pw(_step)

## 「ワークフローを開始」をクリックする

ワークフローが開始されること

In [ ]:
async def _step(page):
    await page.locator('//button[@data-test-workflow-start-button]').click()
    await expect(page.locator('//*[@data-test-workflow-submit-success]')).to_be_visible(timeout=transition_timeout)
    # 通知処理が走るのを待つ
    await asyncio.sleep(10)

await run_pw(_step)

## 「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('#activationsPanel').locator('//button[*[text() = "無効にする"]]').scroll_into_view_if_needed()
    await expect(page.locator('#activationsPanel').locator('//button[*[text() = "無効にする"]]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## アクティベーション一覧で対象行の無効化ボタンをクリックし、ダイアログで確認ボタンをクリックする
ステータスが「無効」に変わること

In [ ]:
async def _step(page):
    await page.locator('#activationsPanel').locator('//button[*[text() = "無効にする"]]').click()
    await expect(page.locator('//h4[text() = "ワークフローの無効化"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[contains(@data-bind, "confirmDisableActivation")]').click()
    await expect(page.locator('#activationsPanel').locator('//*[contains(@class, "label") and text() = "無効"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## アクティベーション一覧で対象行の削除ボタンをクリックする

確認メッセージが表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(@data-bind, "deleteActivation")]').click()
    await expect(page.locator('//h4[text() = "ワークフローの削除"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「削除」をクリックする

エラーメッセージ「Cannot delete activation with running workflows.」が表示されること（実行中のワークフローが存在するため）

In [ ]:
async def _step(page):
    await page.locator('//button[contains(@data-bind, "confirmDeleteActivation")]').click()
    await expect(page.locator('//p[@class="text-danger" and text()="Cannot delete activation with running workflows."]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトトップに戻る

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.goto(project_url_execution)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## タスク一覧の「開く」をクリックする

タスクダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-dashboard').locator('.fa-pencil').click()
    await expect(page.locator('#workflow-field-approval_result')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「結果」ドロップダウンから「承認」を選択する

「承認」が選択されること

In [ ]:
async def _step(page):
    option = page.locator('#workflow-field-approval_result option:has-text("承認")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-field-approval_result').select_option(value=value)

await run_pw(_step)

## 「コメント」に適当な文字列を入力する

コメントが入力されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-field-review_comment').fill('Approved for lifecycle test')

await run_pw(_step)

## 「送信」をクリックする

ワークフローが完了すること

In [ ]:
async def _step(page):
    await page.locator('//button[@data-test-task-dialog-submit]').click()
    await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## アクティベーション一覧で対象行の削除ボタンをクリックする

確認ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(@data-bind, "deleteActivation")]').click()
    await expect(page.locator('//h4[text() = "ワークフローの削除"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 確認ダイアログで「削除」をクリックする

アクティベーションが一覧から削除されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(@data-bind, "confirmDeleteActivation")]').click()
    await expect(page.locator(f'#activationsPanel').locator(f'//td//strong[text()="{workflow_name}"]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードに戻り、テンプレート登録用プロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    global project_url_template
    await page.goto(rdm_url)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name_template}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    project_url_template = page.url

await run_pw(_step)

## 「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('#localWorkflowsPanel').locator('//button[*[text() = "無効にする"]]').scroll_into_view_if_needed()
    await expect(page.locator('#localWorkflowsPanel').locator('//button[*[text() = "無効にする"]]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## テンプレート一覧で対象行の無効化ボタンをクリックする

確認ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#localWorkflowsPanel').locator('//button[*[text() = "無効にする"]]').click()
    await expect(page.locator('//h4[text() = "ワークフローテンプレートの無効化"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 確認ダイアログで確認ボタンをクリックする

テンプレートのステータスが「無効」に変わること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(@data-bind, "confirmDisableTemplate")]').click()
    await expect(page.locator('#localWorkflowsPanel').locator('//*[contains(@class, "label") and text() = "無効"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ワークフロー」セクションを展開する

無効化されたテンプレートがリストに表示されないこと

In [ ]:
async def _step(page):
    await page.locator('//strong[text() = "ワークフロー"]/../i[contains(@class, "fa-chevron-down")]').scroll_into_view_if_needed()
    await page.locator('//strong[text() = "ワークフロー"]/../i[contains(@class, "fa-chevron-down")]').click()
    await expect(page.locator(f'#activate-workflow-select option:has-text("{workflow_name}")')).to_have_count(0, timeout=transition_timeout)
    await asyncio.sleep(2)

await run_pw(_step)

## テンプレート一覧で対象行の有効化ボタンをクリックする

テンプレートのステータスが「有効」に変わること

In [ ]:
async def _step(page):
    await page.locator('#localWorkflowsPanel').locator('//button[*[text() = "有効にする"]]').scroll_into_view_if_needed()
    await page.locator('#localWorkflowsPanel').locator('//button[*[text() = "有効にする"]]').click()
    await expect(page.locator('#localWorkflowsPanel').locator('//*[contains(@class, "label") and text() = "有効"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「更新」をクリックし、テンプレートを選択する

テンプレートがリストに表示されること

In [ ]:
async def _step(page):
    await page.locator('//strong[text() = "ワークフロー"]/../i[contains(@class, "fa-chevron-right")]').scroll_into_view_if_needed()
    await page.locator('//button[contains(@data-bind, "fetchTemplates")]').nth(1).click()
    await expect(page.locator(f'#activate-workflow-select option:has-text("{workflow_name}")')).to_be_attached(timeout=transition_timeout)
    await page.locator('#activate-workflow-select').select_option(label=f'{workflow_name} [{project_name_template}]')

await run_pw(_step)

## テンプレート一覧で対象行の無効化ボタンをクリックし、確認をクリックする

ステータスが「無効」に変わること

In [ ]:
async def _step(page):
    await page.locator('#localWorkflowsPanel').locator('//button[*[text() = "無効にする"]]').scroll_into_view_if_needed()
    await page.locator('#localWorkflowsPanel').locator('//button[*[text() = "無効にする"]]').click()
    await expect(page.locator('//h4[text() = "ワークフローテンプレートの無効化"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[contains(@data-bind, "confirmDisableTemplate")]').click()
    await expect(page.locator('#localWorkflowsPanel').locator('//*[contains(@class, "label") and text() = "無効"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## テンプレート一覧で対象行の削除ボタンをクリックする

確認ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(@data-bind, "deleteTemplate")]').click()
    await expect(page.locator('//h4[text() = "ワークフローテンプレートの削除"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 確認ダイアログで確認ボタンをクリックする

テンプレートが一覧から削除されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(@data-bind, "confirmDeleteTemplate")]').click()
    await expect(page.locator(f'#localWorkflowsPanel').locator(f'//td//strong[text()="{workflow_name}"]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## GakuNin RDM管理者ページのURLを開く

ログイン画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)
    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step, new_page=True)

## ログイン情報を用いてGakuNin RDMにログインする

管理者ダッシュボードが表示されること

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )
    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## サイドメニューから「RDMワークフロー」をクリックする

機関一覧が表示されること

In [ ]:
async def _step(page):
    await page.locator('//span[text()="RDMワークフロー"]').click()
    await expect(page.locator('//table//th[contains(text(), "機関")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 機関一覧から対象の機関名をクリックする

エンジン管理画面が表示されること

In [ ]:
import traceback

async def _step(page):
    while True:
        link = page.locator(f'//table//a[text() = "{institution_name}"]')
        try:
            await expect(link).to_be_visible()
        except:
            traceback.print_exc()
            print('Search next page...')
            await page.locator('.pagination').locator('//a[i[contains(@class, "fa-angle-right")]]').first.click()
            await expect(page.locator('//table//th[contains(text(), "機関")]')).to_be_visible(timeout=transition_timeout)
            continue
        await link.click()
        break
    await expect(page.locator('#id_label')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## エンジン一覧で対象行の無効化ボタンをクリックする

確認ダイアログが表示されること

In [ ]:
async def _step(page):
    row = page.locator(f'//table//tr[.//td//strong[contains(text(), "{engine_name}")]]')
    await row.locator('//button[contains(text(), "無効にする")]').click()
    await expect(page.locator('.modal:visible').locator('//h4[text() = "ワークフローエンジンの無効化"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 確認ダイアログで確認ボタンをクリックする

エンジンのステータスが「無効」に変わること

In [ ]:
async def _step(page):
    await page.locator('.modal:visible').locator('//button[@type="submit"]').click()
    row = page.locator(f'//table//tr[.//td//strong[contains(text(), "{engine_name}")]]')
    await row.locator('//*[contains(@class, "label") and text() = "無効"]').scroll_into_view_if_needed()
    await expect(row.locator('//*[contains(@class, "label") and text() = "無効"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## テンプレート作成プロジェクトを開き、「アドオン」をクリックする

無効化されたエンジンがリストに表示されないこと

In [ ]:
async def _step(page):
    await page.goto(project_url_template)
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[@data-target="#activationsPanel"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'#workflow-engine-id option:has-text("{engine_name}")')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## 管理者ページを開き、「RDMワークフロー」をクリックし、機関名をクリックする

エンジン管理画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)
    await page.locator('//span[text()="RDMワークフロー"]').click()
    await expect(page.locator('//table//th[contains(text(), "機関")]')).to_be_visible(timeout=transition_timeout)
    
    while True:
        link = page.locator(f'//table//a[text() = "{institution_name}"]')
        try:
            await expect(link).to_be_visible()
        except:
            await page.locator('.pagination').locator('//a[i[contains(@class, "fa-angle-right")]]').first.click()
            await expect(page.locator('//table//th[contains(text(), "機関")]')).to_be_visible(timeout=transition_timeout)
            continue
        await link.click()
        break
    await expect(page.locator('#id_label')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## エンジン一覧で対象行の有効化ボタンをクリックする

エンジンのステータスが「有効」に変わること

In [ ]:
async def _step(page):
    row = page.locator(f'//table//tr[.//td//strong[contains(text(), "{engine_name}")]]')
    await row.locator('//button[contains(text(), "有効にする")]').click()
    await expect(row.locator('//*[contains(@class, "label") and text() = "有効"]')).to_be_visible(timeout=transition_timeout)
    await row.locator('//*[contains(@class, "label") and text() = "有効"]').scroll_into_view_if_needed()

await run_pw(_step)

## テンプレート作成プロジェクトを開き、「アドオン」をクリックする

エンジンがリストに表示されること

In [ ]:
async def _step(page):
    await page.goto(project_url_template)
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'#workflow-engine-id option:has-text("{engine_name}")')).to_be_attached(timeout=transition_timeout)
    await page.locator('#workflow-engine-id').scroll_into_view_if_needed()

await run_pw(_step)

## 「ワークフローエンジン」ドロップダウンからエンジンを選択する

指定したエンジンが選択されること

In [ ]:
async def _step(page):
    option = page.locator(f'#workflow-engine-id option:has-text("{engine_name}")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-engine-id').select_option(value=value)

await run_pw(_step)

## 「ワークフローZIP」にBPMNファイルを含むZIPをアップロードする

ZIPファイルが選択されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-zip').set_input_files(workflow_zip_path)

await run_pw(_step)

## 「名前」にワークフロー名を入力する

ワークフロー名が入力されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-label').fill(cascade_test_template_name)

await run_pw(_step)

## 「管理者トークン」ドロップダウンで「閲覧＆編集権限で使用」を選択する

「閲覧＆編集権限で使用」が選択されること

In [ ]:
async def _step(page):
    await page.locator('select[data-bind="value: form.managerTokenMode"]').select_option(value='readwrite')

await run_pw(_step)

## 「ワークフローテンプレートを登録」をクリックする

テンプレートが作成され、有効化用ドロップダウンに表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[.//span[contains(text(), "ワークフローテンプレートを登録")]]').click()
    await expect(page.locator(f'#activate-workflow-select option:has-text("{cascade_test_template_name}")')).to_be_attached(timeout=transition_timeout)

await run_pw(_step)

## 「ワークフロー」を開き、「有効化するテンプレート」ドロップダウンからテンプレートを選択する

指定したテンプレートが選択されること

In [ ]:
async def _step(page):
    await page.locator('//strong[text() = "ワークフロー"]/../i[contains(@class, "fa-chevron-down")]').scroll_into_view_if_needed()
    await page.locator('//strong[text() = "ワークフロー"]/../i[contains(@class, "fa-chevron-down")]').click()
    await page.locator('#activate-workflow-select').scroll_into_view_if_needed()
    option = page.locator(f'#activate-workflow-select option:has-text("{cascade_test_template_name}")')
    value = await option.get_attribute('value')
    await page.locator('#activate-workflow-select').select_option(value=value)

await run_pw(_step)

## 「有効化」をクリックする

「ワークフロー権限の付与」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#activationsPanel').locator('//button[.//span[contains(text(), "有効化")]]').click()
    await expect(page.locator('#activateTokenPermissionModal').locator('//h4[contains(text(), "ワークフロー権限の付与")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「権限を付与して有効化」をクリックする

テンプレートが有効化され、有効化済みテンプレート一覧に表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(text(), "権限を付与して有効化")]').click()
    await expect(page.locator(f'#activationsPanel').locator(f'//td//strong[text()="{cascade_test_template_name}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 管理者ページを開き、「RDMワークフロー」をクリックし、機関名をクリックする

エンジン管理画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)
    await page.locator('//span[text()="RDMワークフロー"]').click()
    await expect(page.locator('//table//th[contains(text(), "機関")]')).to_be_visible(timeout=transition_timeout)
    
    while True:
        link = page.locator(f'//table//a[text() = "{institution_name}"]')
        try:
            await expect(link).to_be_visible()
        except:
            await page.locator('.pagination').locator('//a[i[contains(@class, "fa-angle-right")]]').first.click()
            await expect(page.locator('//table//th[contains(text(), "機関")]')).to_be_visible(timeout=transition_timeout)
            continue
        await link.click()
        break
    await expect(page.locator('#id_label')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## エンジン一覧で対象行の無効化ボタンをクリックする

> TODO: ボタン名・セレクタを確認

ステータスが「無効」に変わること

In [ ]:
async def _step(page):
    row = page.locator(f'//table//tr[.//td//strong[contains(text(), "{engine_name}")]]')
    await row.locator('//button[contains(text(), "無効にする")]').click()
    await expect(page.locator('.modal:visible').locator('//h4[text() = "ワークフローエンジンの無効化"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('.modal:visible').locator('//button[@type="submit"]').click()
    await row.locator('//*[contains(@class, "label") and text() = "無効"]').scroll_into_view_if_needed()
    await expect(row.locator('//*[contains(@class, "label") and text() = "無効"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## エンジン一覧で対象行の削除ボタンをクリックする

確認ダイアログが表示されること（関連するテンプレート・アクティベーションも削除される旨の警告）

In [ ]:
async def _step(page):
    row = page.locator(f'//table//tr[.//td//strong[contains(text(), "{engine_name}")]]')
    await row.locator('//button[contains(@data-target, "deleteEngineModal")]').click()
    await expect(page.locator('.modal:visible').locator('//h4[text() = "ワークフローエンジンの削除"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 確認ダイアログで確認ボタンをクリックする

エンジンが一覧から削除されること

In [ ]:
async def _step(page):
    await page.locator('.modal:visible').locator('//button[@type="submit"]').click()
    await expect(page.locator('//strong[text() = "登録済みワークフローエンジン"]')).to_be_visible(timeout=transition_timeout)
    row = page.locator(f'//table//tr[.//td//strong[contains(text(), "{engine_name}")]]')
    await expect(row).to_have_count(0, timeout=transition_timeout)
    await page.locator('//strong[text() = "登録済みワークフローエンジン"]').scroll_into_view_if_needed()

await run_pw(_step)

## テンプレート作成プロジェクトを開き、「アドオン」をクリックする

関連するワークフローテンプレートも削除されていること

In [ ]:
async def _step(page):
    await page.goto(project_url_template)
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//strong[text() = "ワークフロー"]/../i[contains(@class, "fa-chevron-down")]').scroll_into_view_if_needed()
    await page.locator('#activate-workflow-select').scroll_into_view_if_needed()
    option = page.locator(f'#activate-workflow-select option:has-text("{cascade_test_template_name}")')
    await expect(option).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## 後始末

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}